# 04. LLM.int8(): Mixed-Precision Outlier Decomposition

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Dettmers et al. (2022, *LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale*).
- **Purpose:** Decompose linear transformations into high-precision FP16 operations for the emergent 0.1% outlier channels and INT8 Tensor Core GEMM for the remaining 99.9% of normal channels: $\mathbf{Y} = \mathbf{X}_{\text{outlier}}\mathbf{W}_{\text{outlier}} + \mathbf{X}_{\text{normal}}\mathbf{W}_{\text{normal}}$.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. SmoothQuant and Outlier Migration W8A8](03_smoothquant_and_outlier_migration_w8a8.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Rotation and Incoherence PTQ: QuaRot, SpinQuant and QuIP#](05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use LLM.int8() on models smaller than 6.7B parameters (emergent outliers do not manifest at small scales; pure INT8 or W4A16 is strictly faster); do NOT use a threshold $\tau < 4.0$ (excessive FP16 channels erode INT8 memory and speed benefits).

## Mathematical Foundations & Outlier Matrix Decomposition

1. **Outlier Channel Identification**:
   $$\mathcal{C}_{\text{outlier}} = \{ j \mid \max_i |X_{i, j}| \ge \tau = 6.0 \}, \quad \mathcal{C}_{\text{normal}} = \{ j \mid \max_i |X_{i, j}| < 6.0 \}$$
2. **Additive Matrix Decomposition**:
   $$\mathbf{Y} = \mathbf{X}\mathbf{W} = \mathbf{X}_{:, \mathcal{C}_{\text{outlier}}} \mathbf{W}_{\mathcal{C}_{\text{outlier}}, :} + \mathbf{X}_{:, \mathcal{C}_{\text{normal}}} \mathbf{W}_{\mathcal{C}_{\text{normal}}, :}$$
3. **Vector-Wise INT8 Quantization**:
   $$s_X = \max_j |X_{\text{normal}, i, j}| / 127, \quad s_W = \max_i |W_{\text{normal}, i, j}| / 127$$
4. **Zero Performance Loss**:
   $$k \le 0.1\% \cdot d_{\text{in}} \implies 100\% \text{ FP16 task accuracy with } 50\% \text{ model memory savings}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def quantize_row_int8(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(x), dim=-1, keepdim=True).values / 127.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(x / scale), -128.0, 127.0)
    return q, scale

def quantize_col_int8(w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(w), dim=0, keepdim=True).values / 127.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(w / scale), -128.0, 127.0)
    return q, scale

def llm_int8_linear(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor | None = None, threshold: float = 6.0):
    channel_max = torch.max(torch.abs(x), dim=0).values
    outlier_mask = channel_max >= threshold
    outlier_cols = torch.where(outlier_mask)[0]
    normal_cols = torch.where(~outlier_mask)[0]

    y_fp = x[:, outlier_cols] @ weight[outlier_cols, :] if len(outlier_cols) > 0 else torch.zeros(x.shape[0], weight.shape[1])
    if len(normal_cols) > 0:
        q_x, s_x = quantize_row_int8(x[:, normal_cols])
        q_w, s_w = quantize_col_int8(weight[normal_cols, :])
        y_int8 = (q_x * s_x) @ (q_w * s_w)
    else:
        y_int8 = torch.zeros(x.shape[0], weight.shape[1])

    y_total = y_fp + y_int8
    if bias is not None: y_total = y_total + bias
    return y_total, y_fp, y_int8, outlier_cols.float()

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. Pure INT8 baseline parity
q_x, s_x = quantize_row_int8(weights["x_calib"])
q_w, s_w = quantize_col_int8(weights["w_orig"])
out_pure = (q_x * s_x) @ (q_w * s_w) + weights["b_orig"]
assert float(torch.max(torch.abs(out_pure - weights["out_pure_int8"]))) < 1e-5

# 3. LLM.int8 mixed precision decomposition parity
out_total, out_fp, out_int8, outlier_indices = llm_int8_linear(weights["x_calib"], weights["w_orig"], weights["b_orig"], threshold=6.0)
assert float(torch.max(torch.abs(out_total - weights["out_total"]))) < 1e-5
assert float(torch.max(torch.abs(out_fp - weights["out_fp"]))) < 1e-5
assert float(torch.max(torch.abs(out_int8 - weights["out_int8"]))) < 1e-5
assert float(torch.max(torch.abs(outlier_indices - weights["outlier_indices"]))) < 1e-5
print("All LLM.int8() mixed-precision parity assertions verified!")

In [ ]:
# Comparative SQNR Benchmark: Pure INT8 vs LLM.int8() under Outliers
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_pure = eval_sqnr(weights["out_pure_int8"])
sqnr_decomp = eval_sqnr(weights["out_total"])

print(f"Pure INT8 SQNR: {sqnr_pure:.2f} dB (severely corrupted by outliers)")
print(f"LLM.int8() SQNR: {sqnr_decomp:.2f} dB (gain: +{sqnr_decomp - sqnr_pure:.2f} dB)")
assert sqnr_decomp > sqnr_pure + 10.0, "Outlier decomposition must achieve >10 dB SQNR gain over naive INT8!"

In [ ]:
# Microbenchmark: Parallel Outlier Detection & Column Gathering
b, d_in = 64, 4096
x_mock = torch.randn(b, d_in)
x_mock[:, [12, 105, 3401]] *= 15.0  # 3 outlier channels

t0 = time.perf_counter()
for _ in range(50):
    c_max = torch.max(torch.abs(x_mock), dim=0).values
    out_mask = c_max >= 6.0
    out_cols = torch.where(out_mask)[0]
    x_fp = x_mock[:, out_cols]
t_detect = (time.perf_counter() - t0) / 50 * 1e3

print(f"Outlier Scan & Gather (64x4096) latency: {t_detect:.3f} ms (Found {len(out_cols)} outlier channels)")
assert t_detect < 5.0, "Outlier column scan must complete in <5 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Emergent Outliers**: Scaling beyond 6.7B parameters concentrates critical representation energy into $\le 0.1\%$ of channels.
2. **Decomposition Principle**: Splitting into FP16 outlier and INT8 normal paths guarantees zero perplexity degradation while halving RAM footprint.
3. **Vector-Wise Quantization**: Decoupling rows and columns guarantees optimal scaling for the remaining 99.9% of weights and activations.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_llm_int8_mixed_precision_outlier_decomposition.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.md)
- **PyTorch 2.x Module:** [04_llm_int8_mixed_precision_outlier_decomposition.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.py)
- **Functional JAX Module:** [04_llm_int8_mixed_precision_outlier_decomposition.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.py)
- **Native Rust Candle Track:** [04_llm_int8_mixed_precision_outlier_decomposition.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.rs)
- **Hardware Kernel (CPU / WGSL):** [04_llm_int8_mixed_precision_outlier_decomposition.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.rs)
- **Golden SafeTensors:** [04_llm_int8_mixed_precision_outlier_decomposition.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/04_llm_int8_mixed_precision_outlier_decomposition.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. SmoothQuant and Outlier Migration W8A8](03_smoothquant_and_outlier_migration_w8a8.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 05. Rotation and Incoherence PTQ: QuaRot, SpinQuant and QuIP#](05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.ipynb) &rarr;